In [1]:
import json
from pathlib import Path

from filer_backend.filing.suggester import suggest_folders
from filer_backend.indexing.extract import extract_text

In [2]:
path_str = '/Users/ericmelz/Data/code/filer/apps/backend/evals/EXP04.json'
path = Path(path_str)
path

PosixPath('/Users/ericmelz/Data/code/filer/apps/backend/evals/EXP04.json')

In [3]:
with open(path, 'r') as f:
    data = json.load(f)

In [4]:
imperfect_pdfs = []
for s in data['samples']:
    if s['kind'] == 'pdf' and s['hit@1'] < 1.0:
        imperfect_pdfs.append(s)

In [5]:
len(imperfect_pdfs)

11

In [6]:
len(data['samples'])

50

In [7]:
total_pdfs = len([s for s in data['samples'] if s['kind']  == 'pdf'])
total_pdfs

43

In [8]:
pct_bad = len(imperfect_pdfs) / total_pdfs
pct_bad

0.2558139534883721

In [9]:
i0 = imperfect_pdfs[0]
i0

{'file_id': 286,
 'filename': '2025_08_31.pdf',
 'truth': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements',
 'predicted': ['/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements',
  '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements',
  '/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements'],
 'kind': 'pdf',
 'density': 'existing',
 'hit@1': 0.0,
 'hit@3': 1.0,
 'rr': 0.5,
 'prefix': 0.6666666666666666}

In [10]:
i0['truth']

'/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements'

In [11]:
text, parser = extract_text(Path(i0['truth']) / Path(i0['filename']))
print(text)

Initiate Business Checking SM 
August 31, 2025 Page 1 of 4 
Questions? 
Available by phone Mon-Sat 7:00am-11:00pm Eastern 
Time, Sun 9:00am-10:00pm Eastern Time: 
We accept all relay calls, including 711 
1-800-CALL-WELLS (1-800-225-5935) 
En español: 1-877-337-7454 
Online: wellsfargo.com/biz 
Write: Wells Fargo Bank, N.A. (114) 
P.O. Box 6995 
Portland, OR 97228-6995 
ERIC R MELZ 
RANDI M CURTIS 
DBA 15151 ENCANTO DRIVE 
300 S REEVES DR 
BEVERLY HILLS CA 90212-4513 
Your Business and Wells Fargo 
Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, 
infographics, and other resources on the topics of money movement, account 
management and monitoring, security and fraud prevention, and more.
Other Wells Fargo Benefits
You control your information - Be awarewhat you share
It could be something as innocent as your email address or where you bank or live. Be careful what you share and who you share
it with.
Fraudsters can use your personal information to steal your id

In [12]:
from filer_backend.eval.core import _to_inbox
from filer_backend.storage.db import get_session
from filer_backend.storage.models import File, Folder, InboxFile
from filer_backend.filing.suggester import suggest_folders
from sqlalchemy import select

In [13]:
s = get_session()

In [14]:
q = select(File)

In [15]:
q = q.where(File.id == i0['file_id'])

In [16]:
files = list(s.execute(q).scalars())

In [17]:
files

In [18]:
files[0]

In [19]:
files[0].id

286

In [20]:
f= files[0]

In [21]:
f.absolute_path

'/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements/2025_08_31.pdf'

In [22]:
inbf = _to_inbox(f)

In [23]:
inbf.id

'286'

In [24]:
# Jupyter already runs an asyncio event loop, so blocking calls like
# agent.run_sync() (used by suggest_folders) would otherwise raise
# "This event loop is already running". nest_asyncio makes the loop
# re-entrant so those sync helpers work inside the notebook.
import nest_asyncio

nest_asyncio.apply()

In [25]:
suggestions = suggest_folders(inbf, exclude_file_ids={f.id})

/Users/ericmelz/Data/code/filer/apps/backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
len(suggestions)

3

In [27]:
suggestions[0]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements', confidence=0.9, rationale='This folder contains similar files and is the most relevant destination.', is_new=False)

In [28]:
suggestions[1]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements', confidence=0.8, rationale='This folder also contains similar files and is a good alternative.', is_new=False)

In [29]:
suggestions[2]

Suggestion(folder_path='/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements', confidence=0.7, rationale='This folder is relevant and contains similar files.', is_new=False)

Dive into suggestions inspired by suggester.py code

In [30]:
inbox_file = inbf

In [31]:
path = Path(inbox_file.absolute_path)

In [32]:
text, _ = extract_text(path)

In [33]:
from filer_backend.embedding import get_embedder

In [34]:
emb = get_embedder()

In [35]:
query = text[:4000]

In [36]:
obj = emb.embed([query])

In [37]:
len(obj)

1

In [38]:
qvec = obj[0]

In [39]:
exclude_file_ids = {inbox_file.id}
exclude_file_ids

{'286'}

In [40]:
from filer_backend.filing.retrieval import hybrid_search

In [41]:
hits = hybrid_search(query, qvec, k=20, exclude_file_ids=exclude_file_ids)

In [42]:
len(hits)

20

In [43]:
h0 = hits[0]

### h0 hits spring hill instead of Encanto.  Let's compare vector search vs text search

In [44]:
h0

{'chunk_id': '334:0',
 'file_id': 334,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Initiate Business Checking SM August 31, 2025 Page 1 of 4 Questions? Available by phone Mon-Sat 7:00am-11:00pm Eastern Time, Sun 9:00am-10:00pm Eastern Time: We accept all relay calls, including 711 1-800-CALL-WELLS (1-800-225-5935) En español: 1-877-337-7454 Online: wellsfargo.com/biz Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ RANDI M CURTIS DBA 585 SPRING HILL DRIVE 300 S REEVES DR BEVERLY HILLS CA 90212-4513 Your Business and Wells Fargo Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, infographics, and other resources on the topics of money movement, account management an

In [45]:
from filer_backend.storage import vectors

In [46]:
vhits = vectors.vector_search(qvec, k=20, exclude_file_ids=exclude_file_ids)

In [51]:
thits = vectors.text_search(query, k=20, exclude_file_ids=exclude_file_ids)

In [52]:
v0 = vhits[0]
t0 = thits[0]

In [53]:
v0

{'chunk_id': '334:0',
 'file_id': 334,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Initiate Business Checking SM August 31, 2025 Page 1 of 4 Questions? Available by phone Mon-Sat 7:00am-11:00pm Eastern Time, Sun 9:00am-10:00pm Eastern Time: We accept all relay calls, including 711 1-800-CALL-WELLS (1-800-225-5935) En español: 1-877-337-7454 Online: wellsfargo.com/biz Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ RANDI M CURTIS DBA 585 SPRING HILL DRIVE 300 S REEVES DR BEVERLY HILLS CA 90212-4513 Your Business and Wells Fargo Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, infographics, and other resources on the topics of money movement, account management an

In [54]:
t0

{'chunk_id': '334:0',
 'file_id': 334,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Initiate Business Checking SM August 31, 2025 Page 1 of 4 Questions? Available by phone Mon-Sat 7:00am-11:00pm Eastern Time, Sun 9:00am-10:00pm Eastern Time: We accept all relay calls, including 711 1-800-CALL-WELLS (1-800-225-5935) En español: 1-877-337-7454 Online: wellsfargo.com/biz Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ RANDI M CURTIS DBA 585 SPRING HILL DRIVE 300 S REEVES DR BEVERLY HILLS CA 90212-4513 Your Business and Wells Fargo Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, infographics, and other resources on the topics of money movement, account management an

In [55]:
thits[1]

{'chunk_id': '1091:0',
 'file_id': 1091,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Initiate Business Checking SM August 31, 2025 Page 1 of 4 Questions? Available by phone Mon-Sat 7:00am-11:00pm Eastern Time, Sun 9:00am-10:00pm Eastern Time: We accept all relay calls, including 711 1-800-CALL-WELLS (1-800-225-5935) En español: 1-877-337-7454 Online: wellsfargo.com/biz Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 RANDI M CURTIS ERIC R MELZ DBA WHIFFLETREE DESIGNS 300 S REEVES DR BEVERLY HILLS CA 90212-4513 Your Business and Wells Fargo Visit wellsfargo.com/digitalbusinessresourcesto explore tours, articles, infographics, and other resources on the topics of money movement, account management and monitoring, security and fraud prevent

In [57]:
thits[2]

{'chunk_id': '91:0',
 'file_id': 91,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Joint 0469/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Joint 0469',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Wells Fargo Platinum Savings August 31, 2025 Page 1 of 5 Questions? Available by phone 24 hours a day, 7 days a week: We accept all relay calls, including 711 1-800-TO-WELLS (1-800-869-3557) En español: 1-877-727-2932 Online: wellsfargo.com Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ RANDI M CURTIS 300 S REEVES DR BEVERLY HILLS CA 90212-4513 You and Wells Fargo Thank you for being a loyal Wells Fargo customer. We value your trust in our company and look forward to continuing to serve you with your financial needs. Other Wells Fargo Benefits You control your information - Be awarewhat you share It could be something as innocent 

In [58]:
thits[3]

{'chunk_id': '779:0',
 'file_id': 779,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Wells Fargo Everyday Checking August 31, 2025 Page 1 of 5 Questions? Available by phone 24 hours a day, 7 days a week: We accept all relay calls, including 711 1-800-TO-WELLS (1-800-869-3557) En español: 1-877-727-2932 Online: wellsfargo.com Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ RANDI M CURTIS 300 S REEVES DR BEVERLY HILLS CA 90212-4513 You and Wells Fargo Thank you for being a loyal Wells Fargo customer. We value your trust in our company and look forward to continuing to serve you with your financial needs. Other Wells Fargo Benefits You control your information - Be awarewhat you share It could be som

In [59]:
thits[4]

{'chunk_id': '102:0',
 'file_id': 102,
 'path': '/Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Personal 1737/2025_08_31.pdf',
 'folder_path': '/Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Personal 1737',
 'filename': '2025_08_31.pdf',
 'chunk_index': 0,
 'chunk_text': 'Wells Fargo Platinum Savings August 31, 2025 Page 1 of 5 Questions? Available by phone 24 hours a day, 7 days a week: We accept all relay calls, including 711 1-800-TO-WELLS (1-800-869-3557) En español: 1-877-727-2932 Online: wellsfargo.com Write: Wells Fargo Bank, N.A. (114) P.O. Box 6995 Portland, OR 97228-6995 ERIC R MELZ 300 S REEVES DR BEVERLY HILLS CA 90212-4513 You and Wells Fargo Thank you for being a loyal Wells Fargo customer. We value your trust in our company and look forward to continuing to serve you with your financial needs. Other Wells Fargo Benefits You control your information - Be awarewhat you share It could be something as innocent as your

In [60]:
vids = [f['file_id'] for f in vhits]
tids = [f['file_id'] for f in thits]

In [61]:
vids[:10]

[334, 1091, 288, 283, 287, 337, 330, 338, 1092, 1089]

In [62]:
vfps = [f['folder_path'] for f in vhits]
tfps = [f['folder_path'] for f in thits]

In [65]:
for i, fp in enumerate(vfps):
    print(f'{i:02} {fp}')

00 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
01 /Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking
02 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements
03 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements
04 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements
05 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
06 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
07 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
08 /Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking
09 /Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking
10 /Volumes/home/_Documents/_Records/_By Y

In [68]:
for i, fp in enumerate(tfps):
    print(f'{i:02} {fp}')

00 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
01 /Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking
02 /Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Joint 0469
03 /Volumes/home/_Documents/_Records/_By Year/2025/Reeves/Banking/Wells Fargo Checking/Statements
04 /Volumes/home/_Documents/_Records/_By Year/2025/Banking/Savings/Wells Fargo/Personal 1737
05 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
06 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements
07 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Spring Hill/Banking/Wells Fargo/Statements
08 /Volumes/home/_Documents/_Records/_By Year/2025/Property/Encanto/Banking/Wells Fargo/Statements
09 /Volumes/home/_Documents/_Records/_By Year/2025/Whiffletree/Banking/WF Checking
10 /Volumes/home/_Documents/_Records/_By Year/